In [ ]:
# SPR 2026 – BERTimbau OOF-Weighted Fold Ensemble MAX_LEN=512
# Estratégia: Em vez de média simples dos 5 folds, pesar cada fold pelo
# seu OOF F1-macro individual (extraído de all_results.pkl).
# Folds com melhor OOF F1 contribuem proporcionalmente mais.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import f1_score

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

# Thresholds e temperatura do winner (0.84027)
BEST_TEMP       = 0.9580
BEST_THRESHOLDS = [0.9500, 1.6000, 1.3500, 1.0, 0.4000, 1.1500, 0.1]

print(f'Device: {DEVICE}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET & PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item


INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTRAIR OOF F1 POR FOLD → CALCULAR PESOS
# ═══════════════════════════════════════════════════════════════════════════════
artifacts_path = WEIGHTS_BASE / 'all_results.pkl'
print(f'Carregando artifacts: {artifacts_path}')
with open(artifacts_path, 'rb') as f:
    all_results = pickle.load(f)

fold_f1_scores = {}
for fold_key, fold_data in all_results.items():
    if isinstance(fold_data, dict):
        # Tentar extrair F1 direto ou calcular dos OOF probs
        f1 = fold_data.get('f1_macro', fold_data.get('val_f1', fold_data.get('oof_f1', None)))
        if f1 is None:
            probs  = fold_data.get('val_probs',  fold_data.get('oof_probs',  None))
            labels = fold_data.get('val_labels', fold_data.get('oof_labels', None))
            if probs is not None and labels is not None:
                preds = np.array(probs).argmax(axis=1)
                f1    = f1_score(np.array(labels), preds, average='macro')
        if f1 is not None:
            # Extrair número do fold da chave
            fold_num = int(str(fold_key).split('_')[-1]) if '_' in str(fold_key) else fold_key
            fold_f1_scores[fold_num] = float(f1)

if fold_f1_scores:
    print('OOF F1 por fold:')
    for fold, f1 in sorted(fold_f1_scores.items()):
        print(f'  Fold {fold}: {f1:.5f}')
    # Pesos proporcionais ao F1
    total_f1  = sum(fold_f1_scores.values())
    fold_weights = {fold: f1 / total_f1 for fold, f1 in fold_f1_scores.items()}
    print(f'Pesos normalizados: {fold_weights}')
else:
    # Fallback: pesos iguais
    print('AVISO: OOF F1 não encontrado em all_results.pkl. Usando pesos iguais.')
    fold_weights = {i: 1.0 / N_FOLDS for i in range(N_FOLDS)}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRATION & INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)


# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS DE TESTE
# ═══════════════════════════════════════════════════════════════════════════════
test_path  = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df    = pd.read_csv(test_path)
test_texts = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Test samples: {len(test_df)}')

tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
dataset   = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=True)

config_path = weights_dir / 'model_config'

# ═══════════════════════════════════════════════════════════════════════════════
# WEIGHTED FOLD ENSEMBLE (pesos = OOF F1 por fold)
# ═══════════════════════════════════════════════════════════════════════════════
test_probs   = np.zeros((len(test_df), NUM_CLASSES))
total_weight = 0.0

for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        print(f'Fold {fold} não encontrado, pulando...')
        continue
    w = fold_weights.get(fold, 1.0 / N_FOLDS)
    print(f'Carregando fold {fold} (peso={w:.4f})...', end=' ')
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    fold_probs    = predict(model, loader)
    test_probs   += w * fold_probs
    total_weight += w
    print(f'ok')
    del model, state_dict
    torch.cuda.empty_cache()

# Normalizar caso algum fold tenha sido pulado
test_probs /= total_weight
print(f'Peso total acumulado: {total_weight:.4f}')

# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO + SUBMISSÃO
# ═══════════════════════════════════════════════════════════════════════════════
calibrated_probs = temperature_scale(test_probs, BEST_TEMP)
predictions      = apply_thresholds(calibrated_probs, BEST_THRESHOLDS)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')